In [2]:

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
import pandas
import datasets

import transformers

from sklearn.metrics import precision_recall_fscore_support, accuracy_score

In [5]:
DATA_FILE: str = "/home/s2shsinh/data/processed/DefaktS_Twitter.binary.csv"
TEST_FRAC: float = 0.10

MODEL_SLUG: str = "deepset/gbert-base"

OUT_DIR: str = "./fine_tuning_ouput/"

In [6]:
DATA: pandas.DataFrame = (
    pandas.read_csv(DATA_FILE, index_col=[0])
    .rename(columns={"binary_label": "label"})

    # remove urls
    .pipe(lambda _df: _df.assign(
        text=(
            _df["text"]
            # replace urls with special token
            .str.replace(r"https:\/\/t.co\/\S+", "[URL]", regex=True)
        ),
        label=(
            _df["label"].astype(int)
        )
    ))

    # downsample to smallest category
    .pipe(lambda _df: (
        _df
        .groupby("label")
        .sample(n=min(_df["label"].value_counts()))
    ))
)
DATA.head()

,text,label
id,,
406441,Wieso #Drostenluegt ? Lügt der Drosten etwa? D...,0
408644,Mein Mitgefühl allen Betroffenen der Erdbeben ...,0
408447,"Wie wäre es, wenn Elektrofahrzeuge nur geladen...",0
428407,Sie singen unsere inoffizielle Hymne 🔥♥️🔥 Das ...,0
386974,Die Türkei erlebt derzeit die schwerste Erdbeb...,0


In [7]:
DATA_TRAIN = DATA.sample(frac=1.0 - TEST_FRAC)
DATA_TEST = DATA.loc[DATA.index.difference(DATA_TRAIN.index)]

DATASET_TRAIN = datasets.Dataset.from_pandas(DATA_TRAIN, split="train")
DATASET_TEST = datasets.Dataset.from_pandas(DATA_TEST, split="test")

len(DATASET_TRAIN), len(DATASET_TEST), DATA_TRAIN.label.nunique()

(14805, 1645, 2)

In [8]:
TOKENIZER = transformers.AutoTokenizer.from_pretrained(MODEL_SLUG)
MODEL = transformers.AutoModelForSequenceClassification.from_pretrained(MODEL_SLUG, num_labels=DATA_TRAIN.label.nunique())

/opt/anaconda/envs/transformers4.29.0/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/anaconda/envs/transformers4.29.0/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/anaconda/envs/transformers4.29.0/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/p

In [9]:
def tokenize_function(sample):
    return TOKENIZER(sample["text"], padding="max_length", truncation=True, max_length=512)

In [10]:
train_tokenized_dataset = DATASET_TRAIN.map(tokenize_function, batched=True)
test_tokenized_dataset = DATASET_TEST.map(tokenize_function, batched=True)

Map:   0%|          | 0/14805 [00:00<?, ? examples/s]

Map:   0%|          | 0/1645 [00:00<?, ? examples/s]

In [11]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0.0)
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

trainer = transformers.Trainer(
    model=MODEL,
    args=transformers.TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        output_dir=OUT_DIR,
        overwrite_output_dir=True,
        save_total_limit=1,
        logging_first_step=True,
        logging_steps=50,
        eval_strategy="steps"
    ),
    train_dataset=train_tokenized_dataset,
    eval_dataset=test_tokenized_dataset,
    compute_metrics=compute_metrics,
)

In [12]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
50,0.661400,0.584999,0.689970,0.683843,0.700205,0.687186
100,0.547500,0.575701,0.740426,0.730995,0.790697,0.745014
150,0.455100,0.420934,0.814590,0.814082,0.821203,0.816178
200,0.425100,0.402277,0.827356,0.827133,0.831357,0.828582
250,0.426500,0.370736,0.838906,0.838845,0.840881,0.839768
300,0.408200,0.368795,0.837690,0.837681,0.838573,0.838278
350,0.379400,0.373076,0.831611,0.830961,0.833911,0.830637
400,0.384400,0.367647,0.834043,0.833402,0.836368,0.833069
450,0.363000,0.355359,0.849848,0.849474,0.851072,0.849156
500,0.323700,0.405025,0.840122,0.839598,0.848174,0.841833


TrainOutput(global_step=1389, training_loss=0.29145441830286967, metrics={'train_runtime': 1686.1637, 'train_samples_per_second': 26.341, 'train_steps_per_second': 0.824, 'total_flos': 1.16860775238144e+16, 'train_loss': 0.29145441830286967, 'epoch': 3.0})

In [16]:
predictions = trainer.predict(test_tokenized_dataset)

In [17]:
from sklearn.metrics import accuracy_score, f1_score



preds = predictions.predictions.argmax(-1)  # If your model outputs logits, take argmax

# Get the true labels
labels = predictions.label_ids

In [18]:
accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average='weighted')  # Adjust average based on your use case

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.8419
F1 Score: 0.8419
